# SketchImpact — Collision Classifier Training

In [ ]:
# Config
DATASET    = 'dataset.npz'
OUTPUT     = 'model_best.pt'

EPOCHS     = 20
BATCH_SIZE = 512
LR         = 1e-3
WORKERS    = 2

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from IPython.display import clear_output

from dataset_loader import CollisionDataset
from model import CollisionCNN

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

train_ds = CollisionDataset(DATASET, split='train')
val_ds   = CollisionDataset(DATASET, split='val')
print(f'Train: {len(train_ds):,}  Val: {len(val_ds):,}')

train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=WORKERS, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=WORKERS, pin_memory=True)

model     = CollisionCNN().to(device)
optimiser = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
loss_fn   = nn.BCEWithLogitsLoss()

In [ ]:
def _metrics(logits, labels):
    preds  = (logits > 0).long()
    labels = labels.long()
    tp = ((preds == 1) & (labels == 1)).sum().item()
    fp = ((preds == 1) & (labels == 0)).sum().item()
    fn = ((preds == 0) & (labels == 1)).sum().item()
    tn = ((preds == 0) & (labels == 0)).sum().item()
    acc  = (tp + tn) / len(labels)
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
    return acc, prec, rec, f1

def plot_history(history):
    epochs = range(1, len(history['train_loss']) + 1)
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    axes[0].plot(epochs, history['train_loss'], label='train')
    axes[0].plot(epochs, history['val_loss'],   label='val')
    axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend()

    axes[1].plot(epochs, history['train_f1'], label='train')
    axes[1].plot(epochs, history['val_f1'],   label='val')
    axes[1].set_title('F1'); axes[1].set_xlabel('Epoch'); axes[1].legend()

    axes[2].plot(epochs, history['train_recall'], label='train')
    axes[2].plot(epochs, history['val_recall'],   label='val')
    axes[2].set_title('Recall'); axes[2].set_xlabel('Epoch'); axes[2].legend()

    plt.tight_layout()
    plt.show()

In [ ]:
history = {'train_loss': [], 'val_loss': [],
           'train_f1': [],   'val_f1': [],
           'train_recall': [],'val_recall': []}

best_val_loss = float('inf')

for epoch in range(1, EPOCHS + 1):
    # --- train ---
    model.train()
    tr_loss, tr_logits, tr_labels = 0.0, [], []
    for x, label in train_dl:
        x, label = x.to(device), label.to(device)
        optimiser.zero_grad()
        logit = model(x)
        loss  = loss_fn(logit, label)
        loss.backward()
        optimiser.step()
        tr_loss += loss.item() * len(label)
        tr_logits.append(logit.detach().cpu())
        tr_labels.append(label.cpu())
    tr_loss /= len(train_ds)
    tr_acc, tr_p, tr_r, tr_f1 = _metrics(torch.cat(tr_logits), torch.cat(tr_labels))

    # --- val ---
    model.eval()
    vl_loss, vl_logits, vl_labels = 0.0, [], []
    with torch.no_grad():
        for x, label in val_dl:
            x, label = x.to(device), label.to(device)
            logit = model(x)
            vl_loss += loss_fn(logit, label).item() * len(label)
            vl_logits.append(logit.cpu())
            vl_labels.append(label.cpu())
    vl_loss /= len(val_ds)
    vl_acc, vl_p, vl_r, vl_f1 = _metrics(torch.cat(vl_logits), torch.cat(vl_labels))

    # --- record ---
    history['train_loss'].append(tr_loss);    history['val_loss'].append(vl_loss)
    history['train_f1'].append(tr_f1);        history['val_f1'].append(vl_f1)
    history['train_recall'].append(tr_r);     history['val_recall'].append(vl_r)

    if vl_loss < best_val_loss:
        best_val_loss = vl_loss
        torch.save({'epoch': epoch, 'model': model.state_dict(), 'val_loss': vl_loss}, OUTPUT)
        saved = '  ← saved'
    else:
        saved = ''

    print(f'Epoch {epoch:02d}/{EPOCHS} | '
          f'train loss={tr_loss:.4f} acc={tr_acc:.3f} P={tr_p:.3f} R={tr_r:.3f} F1={tr_f1:.3f} | '
          f'val   loss={vl_loss:.4f} acc={vl_acc:.3f} P={vl_p:.3f} R={vl_r:.3f} F1={vl_f1:.3f}{saved}')

    clear_output(wait=True)
    plot_history(history)

print(f'\nDone. Best val loss: {best_val_loss:.4f}')

In [ ]:
!python eval.py --checkpoint model_best.pt --dataset dataset.npz